## ___Parse the mycorrhizal state `Table 1` from Wang & Qiu (2008)___
---------------------------

In [1]:
!python --version

Python 3.14.5


The system cannot find the path specified.


In [75]:
import re
from collections import namedtuple

from bs4 import BeautifulSoup
import numpy as np
import pandas as pd

In [7]:
# the paper is behind a paywall so direct scraping won't work - had to log in with uni email, download the html fom Firefox in order to get the table
# https://link.springer.com/article/10.1007/s00572-005-0033-6/tables/1
# https://link.springer.com/article/10.1007/s00572-005-0033-6

with open(file=r"../../data/chapter2/Wang & Qiu - 2006 - Table1.html", mode="rt", encoding="utf8") as fp:
    soup = BeautifulSoup(fp.read())

In [10]:
table = soup.find("table", attrs={"class": "data last-table"}) # the whole mycorrhizal state table

In [25]:
table.find_all("tr")[10]

<tr><td><p>  <i>Marchantia foliacea</i>
</p></td><td><p>AM-like</p></td><td><p>505</p></td></tr>

In [26]:
table.find_all("tr")[10].find_all("td")

[<td><p>  <i>Marchantia foliacea</i>
 </p></td>,
 <td><p>AM-like</p></td>,
 <td><p>505</p></td>]

In [73]:
# species name, mycorrhizal state & references
[(tr.find("i").text, tr.find_all("td")[1].text, tr.find_all("td")[2].text) for tr in table.find_all("tr") if len(tr.find_all("td")) > 2][:20]

[('Haplomitrium gibbsiae', 'AM-like', '107'),
 ('Haplomitrium ovalifolium', 'AM-like', '107'),
 ('Blasia pusilla', 'None', '182, 328'),
 ('Lunularia cruciata', 'Fungal association (G)', '182'),
 ('Marchantia foliacea', 'AM-like', '505'),
 ('Marchantia polymorpha', 'Fungal association (G)', '182'),
 ('Asterella wilmsii', 'AM-like', '345'),
 ('Conocephalum conicum', 'AM-like', '346'),
 ('Riccia fluitans', 'None', '182'),
 ('Apometzgeria pubescens', 'None', '466'),
 ('Metzgeria conjugata', 'None', '466'),
 ('Metzgeria fruticulosa', 'None', '466'),
 ('Metzgeria furcata', 'None', '466'),
 ('Metzgeria leptoneura', 'None', '466'),
 ('Metzgeria temperata', 'None', '466'),
 ('Aneura pinguis', 'ORM-like (B)', '347'),
 ('Cryptothallus mirabilis', 'Fungal association', '466'),
 ('Riccardia latifrons', 'None', '466'),
 ('Riccardia multifida', 'None', '466'),
 ('Riccardia palmate', 'None', '466')]

In [72]:
[tr for tr in table.find_all("tr") if len(tr.find_all("td")) < 2][:20]

[<tr><th><p>Examined species<sup>a</sup>
 </p></th><th><p>Mycorrhizal status<sup>b,c</sup>
 </p></th><th><p>References</p></th></tr>,
 <tr><td class="u-text-center" colspan="3"><p>BRYOPHYTES</p></td></tr>,
 <tr><td colspan="3"><p> Haplomitriaceae</p></td></tr>,
 <tr><td colspan="3"><p> Blasiaceae</p></td></tr>,
 <tr><td colspan="3"><p> Lunulariaceae</p></td></tr>,
 <tr><td colspan="3"><p> Marchantiaceae</p></td></tr>,
 <tr><td colspan="3"><p> Aytoniaceae</p></td></tr>,
 <tr><td colspan="3"><p> Conocephalaceae</p></td></tr>,
 <tr><td colspan="3"><p> Ricciaceae</p></td></tr>,
 <tr><td colspan="3"><p> Metzgeriaceae</p></td></tr>,
 <tr><td colspan="3"><p> Aneuraceae</p></td></tr>,
 <tr><td colspan="3"><p> Pelliaceae</p></td></tr>,
 <tr><td colspan="3"><p> Codoniaceae</p></td></tr>,
 <tr><td colspan="3"><p> Radulaceae</p></td></tr>,
 <tr><td colspan="3"><p> Porellaceae</p></td></tr>,
 <tr><td colspan="3"><p> Jubulaceae</p></td></tr>,
 <tr><td colspan="3"><p> Lejeuneaceae</p></td></tr>,
 <tr

In [74]:
table.find_all("td", attrs={"colspan": '3'})[:20]

[<td class="u-text-center" colspan="3"><p>BRYOPHYTES</p></td>,
 <td colspan="3"><p> Haplomitriaceae</p></td>,
 <td colspan="3"><p> Blasiaceae</p></td>,
 <td colspan="3"><p> Lunulariaceae</p></td>,
 <td colspan="3"><p> Marchantiaceae</p></td>,
 <td colspan="3"><p> Aytoniaceae</p></td>,
 <td colspan="3"><p> Conocephalaceae</p></td>,
 <td colspan="3"><p> Ricciaceae</p></td>,
 <td colspan="3"><p> Metzgeriaceae</p></td>,
 <td colspan="3"><p> Aneuraceae</p></td>,
 <td colspan="3"><p> Pelliaceae</p></td>,
 <td colspan="3"><p> Codoniaceae</p></td>,
 <td colspan="3"><p> Radulaceae</p></td>,
 <td colspan="3"><p> Porellaceae</p></td>,
 <td colspan="3"><p> Jubulaceae</p></td>,
 <td colspan="3"><p> Lejeuneaceae</p></td>,
 <td colspan="3"><p> Pseudolepicoleaceae</p></td>,
 <td colspan="3"><p> Herbertaceae</p></td>,
 <td colspan="3"><p> Plagiochilaceae</p></td>,
 <td colspan="3"><p> Arnelliaceae</p></td>]

In [56]:
# we don't want the GYNMOSPERMS, ANGIOSPERMS.... rows
# <td class="u-text-center" colspan="3"><p>ANGIOSPERMS</p></td>

[td.find('p').text.strip() for td in table.find_all("td", attrs={"colspan": '3', "class": None})]

['Haplomitriaceae',
 'Blasiaceae',
 'Lunulariaceae',
 'Marchantiaceae',
 'Aytoniaceae',
 'Conocephalaceae',
 'Ricciaceae',
 'Metzgeriaceae',
 'Aneuraceae',
 'Pelliaceae',
 'Codoniaceae',
 'Radulaceae',
 'Porellaceae',
 'Jubulaceae',
 'Lejeuneaceae',
 'Pseudolepicoleaceae',
 'Herbertaceae',
 'Plagiochilaceae',
 'Arnelliaceae',
 'Geocalycaceae',
 'Lepidoziaceae',
 'Scapaniaceae',
 'Cephaloziellaceae',
 'Cephaloziaceae',
 'Calypogeiaceae',
 'Jungermanniaceae',
 'Gymnomitriaceae',
 'Anthocerotaceae',
 'Lycopodiaceae',
 'Isoetaceae',
 'Selaginellaceae',
 'Equisetaceae',
 'Marattiaceae',
 'Psilotaceae',
 'Ophioglossaceae',
 'Osmundaceae',
 'Hymenophyllaceae',
 'Gleicheniaceae',
 'Schizaeaceae',
 'Marsileaceae',
 'Azollaceae',
 'Plagiogyriaceae',
 'Cyatheaceae',
 'Dicksoniaceae',
 'Pteridaceae',
 'Vittariaceae',
 'Adiantaceae',
 'Dennstaedtiaceae',
 'Aspleniaceae',
 'Thelypteridaceae',
 'Blechnaceae',
 'Dryopteridaceae',
 'Nephrolepidaceae',
 'Polypodiaceae',
 'Grammitidaceae',
 'Davalliaceae

In [77]:
table_row = namedtuple(typename="table_row", field_names=("family", "examined_species", "mycorrhizal_status", "references"))

In [88]:
# families = dict.fromkeys([td.find('p').text.strip() for td in table.find_all("td", attrs={"colspan": '3', "class": None})], [])
records = []

for row in table.find_all("tr"):
    if row.find("td", attrs={"colspan": '3', "class": None}):
        # print(row.find('p').text) # family name
        fam = row.find('p').text.strip() # these names contain a preceeding space, hence the string.strip()
        assert (fam in families), "family name must be in the dict!"
        continue
        
    elif len(row.find_all("td")) == 3: # binominal name, mycorrhizal state & reference id
        # print(row.find("i").text, row.find_all("td")[1].text, row.find_all("td")[2].text)
        # families[fam].append((row.find("i").text, row.find_all("td")[1].text, row.find_all("td")[2].text))
        records.append(table_row(family=fam,
                                 examined_species=row.find("i").text.strip(),
                                 mycorrhizal_status=row.find_all("td")[1].text.strip(),
                                 references=row.find_all("td")[2].text)
                      )

In [81]:
wang_n_qiu_2006 = pd.DataFrame(records)
wang_n_qiu_2006

,family,examined_species,mycorrhizal_status,references
0,Haplomitriaceae,Haplomitrium gibbsiae,AM-like,107
1,Haplomitriaceae,Haplomitrium ovalifolium,AM-like,107
2,Blasiaceae,Blasia pusilla,None,"182, 328"
3,Lunulariaceae,Lunularia cruciata,Fungal association (G),182
4,Marchantiaceae,Marchantia foliacea,AM-like,505
...,...,...,...,...
3611,Sapindaceae,Matayba guianensis,AM,31
3612,Sapindaceae,Pometia tomentosa,AM,422
3613,Sapindaceae,Sapindus saponaria,AM,548
3614,Bruniaceae,Staavia radiata,AM,24


In [86]:
wang_n_qiu_2006.sort_values(by=["family", "examined_species"], inplace=True)

In [87]:
wang_n_qiu_2006.to_csv(r"../../data/chapter2/Wang & Qiu - 2006 - Table1.csv", index=False)